# FILE 3/3 — Train luồng A + Đanh gia A & B + Bang so sanh 3 seed (seed = 1712)

Notebook nay hoan tat ablation study voi seed 1712:
1. **Train luồng A:** COCO → HCMC truc tiep → model A.
2. **Danh gia model A** tren 3 tap test: HCMC, COCO, CanTho.
3. **Danh gia model B** (tu FILE 2) tren 3 tap test.
4. **Bang so sanh** A vs B, va **doi chieu voi ket qua seed 42 va seed 1407** da co,
   ket luan xu huong co on dinh qua ca 3 seed hay khong.

**BAT BUOC chay xong FILE 1 va FILE 2 truoc.**

> **LUU Y QUAN TRONG NEU CHAY BANG TAI KHOAN DUOC CHIA SE (vi du tai khoan cua Tai):**
> Thu muc "Do an AI giao thong" duoc share se nam o muc **"Shared with me"**, KHONG nam trong
> "My Drive", nen duong dan `/content/drive/MyDrive/Do an AI giao thong/...` se KHONG tim thay.
>
> **Cach xu ly (chi lam 1 lan, khong can sua code):**
> 1. Mo Google Drive tren web, vao muc **"Shared with me"**.
> 2. Tim thu muc **"Do an AI giao thong"**, chuot phai → **Organize** → **Add shortcut**.
> 3. Chon vi tri **"My Drive"** (thu muc goc) → bam **Add**.
>
> Sau buoc nay, moi duong dan trong notebook se hoat dong binh thuong.
> Cell "Buoc 3" ben duoi co dong kiem tra ton tai, neu bao False thi lam lai buoc tren.


## Buoc 1: Cai dat thu vien

In [3]:
!pip install ultralytics -q
print("Da cai dat xong ultralytics")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 8.1 MB/s eta 0:00:00
Da cai dat xong ultralytics


## Buoc 2: Ket noi Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Buoc 3: Khai bao duong dan va cau hinh

In [5]:
import os

SEED = 1712

BASELINE_COCO = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt"
MODEL_B_FINAL = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_hcmc_from_cantho_seed1712/weights/best.pt"

DRIVE_DIR_HCMC = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515"
DRIVE_DIR_COCO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515"
DATA_YAML_CANTHO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515/data.yaml"

LOCAL_DIR_HCMC = "/content/hcmc_701515"
LOCAL_DIR_COCO = "/content/coco_701515"

PROJECT_DIR = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline"
NAME_A = "finetune_hcmc_from_coco_seed1712"

IMGSZ, BATCH, EPOCHS, PATIENCE, LR0 = 640, 16, 100, 20, 0.001

print("Kiem tra su ton tai:")
print("  Baseline COCO          :", os.path.exists(BASELINE_COCO))
print("  Model B cuoi (FILE 2)  :", os.path.exists(MODEL_B_FINAL))
print("  Dataset HCMC (Drive)   :", os.path.exists(DRIVE_DIR_HCMC))
print("  Dataset COCO (Drive)   :", os.path.exists(DRIVE_DIR_COCO))
print("  Yaml CanTho            :", os.path.exists(DATA_YAML_CANTHO))
if not os.path.exists(MODEL_B_FINAL):
    print("\n[LOI] Chua tim thay model B cuoi cung. Hay chay xong FILE 2 truoc!")

Kiem tra su ton tai:
  Baseline COCO          : True
  Model B cuoi (FILE 2)  : True
  Dataset HCMC (Drive)   : True
  Dataset COCO (Drive)   : True
  Yaml CanTho            : True


## Buoc 4: Copy dataset HCMC va COCO ve o cung local

In [7]:
import shutil, time

def copy_ds(drive_dir, local_dir, ten):
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    t0 = time.time()
    try:
        shutil.copytree(drive_dir, local_dir)
        print(f"[OK] Da copy {ten} ve {local_dir} ({time.time()-t0:.1f}s)")
    except shutil.Error as e:
        print(f"[CANH BAO] {len(e.args[0])} file loi tam thoi khi copy {ten}.")

copy_ds(DRIVE_DIR_HCMC, LOCAL_DIR_HCMC, "HCMC")
copy_ds(DRIVE_DIR_COCO, LOCAL_DIR_COCO, "Vehicles-COCO")

DATA_YAML_HCMC_LOCAL = f"{LOCAL_DIR_HCMC}/data.yaml"
DATA_YAML_COCO_LOCAL = f"{LOCAL_DIR_COCO}/data.yaml"
print()
print("Yaml HCMC (local):", os.path.exists(DATA_YAML_HCMC_LOCAL))
print("Yaml COCO (local):", os.path.exists(DATA_YAML_COCO_LOCAL))

[OK] Da copy HCMC ve /content/hcmc_701515 (849.2s)
[OK] Da copy Vehicles-COCO ve /content/coco_701515 (1003.0s)

Yaml HCMC (local): True
Yaml COCO (local): True


## Buoc 5: Huan luyen luồng A — COCO → HCMC truc tiep (seed 1712)

Khoi tao tu Baseline COCO, fine-tune thang tren HCMC (bo qua CanTho). Mat khoang 1-1.5 gio.

In [ ]:
from ultralytics import YOLO

model_A = YOLO(BASELINE_COCO)
results_A = model_A.train(
    data=DATA_YAML_HCMC_LOCAL,
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    lr0=LR0, seed=SEED,
    project=PROJECT_DIR, name=NAME_A, exist_ok=True, verbose=True,
)
print("\nDa huan luyen xong luong A (COCO -> HCMC).")
BEST_A = os.path.join(PROJECT_DIR, NAME_A, "weights", "best.pt")
print("Model A luu tai:", BEST_A)
print("Ton tai:", os.path.exists(BEST_A))

New https://pypi.org/project/ultralytics/8.4.149 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.148 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hcmc_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/cont

##Buoc 5.1 Tiep tuc huan luyen model A sau khi Colab bi ngat**

In [5]:
from ultralytics import YOLO
import os

LAST_A = os.path.join(
    PROJECT_DIR,
    NAME_A,
    "weights",
    "last.pt"
)

print("Checkpoint:", LAST_A)
print("Ton tai:", os.path.exists(LAST_A))

if not os.path.exists(LAST_A):
    raise FileNotFoundError("Khong tim thay last.pt")

model_A = YOLO(LAST_A)
results_A = model_A.train(resume=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Checkpoint: /content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_hcmc_from_coco_seed1712/weights/last.pt
Ton tai: True
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hcmc_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=F

## Buoc 6: Danh gia model A tren 3 tap test

In [11]:
from ultralytics import YOLO
import os
BEST_A = os.path.join(PROJECT_DIR, NAME_A, "weights", "best.pt")
model_A_eval = YOLO(BEST_A)

mA_hcmc = model_A_eval.val(data=DATA_YAML_HCMC_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_A + "_eval_hcmc", exist_ok=True)
A_hcmc_map50, A_hcmc_map = float(mA_hcmc.box.map50), float(mA_hcmc.box.map)

mA_coco = model_A_eval.val(data=DATA_YAML_COCO_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_A + "_eval_coco", exist_ok=True)
A_coco_map50, A_coco_map = float(mA_coco.box.map50), float(mA_coco.box.map)

mA_cantho = model_A_eval.val(data=DATA_YAML_CANTHO, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_A + "_eval_cantho", exist_ok=True)
A_cantho_map50, A_cantho_map = float(mA_cantho.box.map50), float(mA_cantho.box.map)

print("===== MODEL A (COCO->HCMC) =====")
print(f"Test HCMC   : mAP@0.5={A_hcmc_map50:.4f}  mAP@0.5:0.95={A_hcmc_map:.4f}")
print(f"Test COCO   : mAP@0.5={A_coco_map50:.4f}  mAP@0.5:0.95={A_coco_map:.4f}")
print(f"Test CanTho : mAP@0.5={A_cantho_map50:.4f}  mAP@0.5:0.95={A_cantho_map:.4f}")

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.0 ms, read: 35.1±14.0 MB/s, size: 48.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/.shortcut-targets-by-id/1WhcqICO5w6iVwbBcCYkPMXQf9-IvVZMs/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/labels... 2267 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2267/2267 151.0it/s 15.0s
val: /content/drive/.shortcut-targets-by-id/1WhcqICO5w6iVwbBcCYkPMXQf9-IvVZMs/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/images/cam_01_000596_jpg.rf.e27909b3bca621ade420095897f7c5e7.jpg: 1 duplicate labels removed
val: /content/drive/.shortcut-targets-by-id/1WhcqICO5w6iVwbBcCYkPMXQf9-IvVZMs/Do an A

## Buoc 7: Danh gia model B tren 3 tap test

In [8]:
from ultralytics import YOLO
import os

model_B_eval = YOLO(MODEL_B_FINAL)

mB_hcmc = model_B_eval.val(data=DATA_YAML_HCMC_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name="B_" + NAME_A.split("_")[-1] + "_eval_hcmc", exist_ok=True)
B_hcmc_map50, B_hcmc_map = float(mB_hcmc.box.map50), float(mB_hcmc.box.map)

mB_coco = model_B_eval.val(data=DATA_YAML_COCO_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name="B_" + NAME_A.split("_")[-1] + "_eval_coco", exist_ok=True)
B_coco_map50, B_coco_map = float(mB_coco.box.map50), float(mB_coco.box.map)

mB_cantho = model_B_eval.val(data=DATA_YAML_CANTHO, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name="B_" + NAME_A.split("_")[-1] + "_eval_cantho", exist_ok=True)
B_cantho_map50, B_cantho_map = float(mB_cantho.box.map50), float(mB_cantho.box.map)

print("===== MODEL B (COCO->CanTho->HCMC) =====")
print(f"Test HCMC   : mAP@0.5={B_hcmc_map50:.4f}  mAP@0.5:0.95={B_hcmc_map:.4f}")
print(f"Test COCO   : mAP@0.5={B_coco_map50:.4f}  mAP@0.5:0.95={B_coco_map:.4f}")
print(f"Test CanTho : mAP@0.5={B_cantho_map50:.4f}  mAP@0.5:0.95={B_cantho_map:.4f}")

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.1 ms, read: 42.2±19.4 MB/s, size: 62.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/.shortcut-targets-by-id/1WhcqICO5w6iVwbBcCYkPMXQf9-IvVZMs/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/labels.cache... 2267 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2267/2267 339.6Mit/s 0.0s
val: /content/drive/.shortcut-targets-by-id/1WhcqICO5w6iVwbBcCYkPMXQf9-IvVZMs/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/images/cam_01_000596_jpg.rf.e27909b3bca621ade420095897f7c5e7.jpg: 1 duplicate labels removed
val: /content/drive/.shortcut-targets-by-id/1WhcqICO5w6iVwbBcCYkPMXQf9-IvVZMs/D

## Buoc 8: Bang so sanh + doi chieu 3 seed (42, 1407, 1712)

Bang tong hop A vs B voi seed 1712, dat canh ket qua 2 seed truoc do, de xac nhan xu huong
"hai lo trinh tuong duong" co on dinh qua ca ba lan chay hay khong.

In [9]:
# ===== Ket qua 2 seed truoc do (de doi chieu) =====
A42   = {"hcmc":0.9539, "coco":0.0931, "cantho":0.7635}
B42   = {"hcmc":0.9530, "coco":0.0843, "cantho":0.7500}
A1407 = {"hcmc":0.9550, "coco":0.0959, "cantho":0.7757}
B1407 = {"hcmc":0.9545, "coco":0.0659, "cantho":0.7779}

print("="*84)
print(f"BANG SO SANH ABLATION - SEED 1712  (mAP@0.5)")
print("="*84)
print(f"{'Phep danh gia':28s} {'A (COCO->HCMC)':>16s} {'B (COCO->CT->HCMC)':>20s} {'Chenh (A-B)':>12s}")
print("-"*84)
for ten, a, b in [
    ("Test HCMC (in-domain)", A_hcmc_map50, B_hcmc_map50),
    ("Test COCO (forgetting)", A_coco_map50, B_coco_map50),
    ("Test CanTho", A_cantho_map50, B_cantho_map50),
]:
    print(f"{ten:28s} {a:>16.4f} {b:>20.4f} {a-b:>+12.4f}")
print("="*84)

print()
print("="*84)
print("DOI CHIEU 3 SEED - mAP@0.5 tren TEST HCMC (chi tieu quyet dinh)")
print("="*84)
print(f"{'':22s} {'Seed 42':>12s} {'Seed 1407':>12s} {'Seed 1712':>12s}")
print(f"{'A (COCO->HCMC)':22s} {A42['hcmc']:>12.4f} {A1407['hcmc']:>12.4f} {A_hcmc_map50:>12.4f}")
print(f"{'B (COCO->CT->HCMC)':22s} {B42['hcmc']:>12.4f} {B1407['hcmc']:>12.4f} {B_hcmc_map50:>12.4f}")
d42, d1407 = A42['hcmc']-B42['hcmc'], A1407['hcmc']-B1407['hcmc']
d_new = A_hcmc_map50 - B_hcmc_map50
print(f"{'Chenh lech (A-B)':22s} {d42:>+12.4f} {d1407:>+12.4f} {d_new:>+12.4f}")
print("="*84)

print()
print("--- KET LUAN TU DONG ---")
all_d = [d42, d1407, d_new]
if all(abs(d) < 0.01 for d in all_d):
    print(f"Ca BA seed deu cho chenh lech A-B tren test HCMC < 0.01:")
    print(f"  seed 42 = {d42:+.4f} | seed 1407 = {d1407:+.4f} | seed 1712 = {d_new:+.4f}")
    print()
    print("=> Ket luan 'hai lo trinh cho hieu nang TUONG DUONG tren mien dich HCMC'")
    print("   duoc xac nhan on dinh qua ba lan chay doc lap voi ba seed khac nhau.")
    print("   Do do viec bo qua buoc trung gian CanTho la co co so.")
else:
    print("Co seed cho chenh lech >= 0.01, xu huong khong hoan toan on dinh.")
    print(f"  seed 42 = {d42:+.4f} | seed 1407 = {d1407:+.4f} | seed 1712 = {d_new:+.4f}")
    print("=> Nen trinh bay ca ba ket qua mot cach minh bach trong bao cao.")

print()
print("--- LUU Y ve TEST CANTHO (can trinh bay minh bach) ---")
dc42, dc1407 = A42['cantho']-B42['cantho'], A1407['cantho']-B1407['cantho']
dc_new = A_cantho_map50 - B_cantho_map50
print(f"  seed 42 = {dc42:+.4f} | seed 1407 = {dc1407:+.4f} | seed 1712 = {dc_new:+.4f}")
if not (dc42 > 0) == (dc1407 > 0) or not (dc1407 > 0) == (dc_new > 0):
    print("  => Chieu chenh lech DAO NGUOC giua cac seed, cho thay khac biet tren mien")
    print("     CanTho nam trong pham vi nhieu ngau nhien, khong phai quy luat.")

BANG SO SANH ABLATION - SEED 1712  (mAP@0.5)
Phep danh gia                  A (COCO->HCMC)   B (COCO->CT->HCMC)  Chenh (A-B)
------------------------------------------------------------------------------------


NameError: name 'A_hcmc_map50' is not defined